This notebook is a template for generating a fast PSF emulator.

In [ ]:
import time
from pathlib import Path

import numpy as np

from matplotlib import pyplot as plt

from astropy import units as u

from jwst import datamodels

from nirspectroperf.psf import NIRSpecPSF, NIRSpecPSFINREmulator

import jax
from jax import numpy as jnp

In [ ]:
random_key_seed = 42  # can be None to use the time

cache_fn = 'psf_cache.npz'  # None to not use the cache

device_dtype = jnp.float32

psf_oversampling = 4
psf_fov = 50
npsfs = 1024

grid_type = 'random' # 'random' or 'grid'

learning_rate = 2e-4
nepochs = 25000
batch_size = 2048

save_model_fn_pattern = 'nirspec_psf_estimator_{dt}.hdf5'  # use "{dt}" to get a timestamp

In [ ]:
if random_key_seed is None:
    random_key_seed = time.time_ns()
jrng = jax.random.key(random_key_seed)

cache_path = Path(cache_fn) if cache_fn is not None else None

In [ ]:
n6791fits = list(Path('../../ngc6791_cals').glob('*.fits'))
dm0 = datamodels.open(n6791fits[0])
slit = dm0.slits[49]

In [ ]:
# Use this to bust the cache and force regeneration of the PSF if needed

if cache_path.exists():
    cache_path.unlink()

In [ ]:
if cache_path is None or not cache_path.is_file():
    nrs_psf = NIRSpecPSF(observation_template_dm=dm0, 
                        slit=slit, 
                        oversampling=psf_oversampling, 
                        fov=psf_fov)
    if grid_type == 'random':
        psfs, xpx, ypx, xos, yos = nrs_psf.make_psf_points(npsfs, 1*u.micron)
    elif grid_type == 'grid':
        psfs, xpx, ypx, xos, yos = nrs_psf.make_psf_grid(npsfs, 1*u.micron)
    else:
        raise ValueError(f'Unknown grid_type: {grid_type}')
    psfs = np.array(psfs, dtype=device_dtype)
    xpx = np.array(xpx, dtype=device_dtype)
    ypx = np.array(ypx, dtype=device_dtype)
    psfxs = np.array(xos, dtype=device_dtype)
    psfys = np.array(yos, dtype=device_dtype)

    if cache_path is not None:
        print('Saving to cache file', cache_path)
        np.savez(cache_path, psflux=psfs, xpx=xpx, ypx=ypx, xoffs=psfxs, yoffs=psfys,
                 oversampling=psf_oversampling, fov=psf_fov, grid_type=grid_type)

print('Loading from cache file', cache_path)
cache = np.load(cache_path, allow_pickle=True)

if len(cache['psflux']) != npsfs:
    raise ValueError('loaded/generated PSF is the wrong size!  Probably need to delete an old cache')
if cache['oversampling'] != psf_oversampling or cache['fov'] != psf_fov or cache['grid_type'] != grid_type:
    raise ValueError('loaded/generated PSF has different oversampling or FOV than expected!  Probably need to delete an old cache')

psfs = jnp.array(cache['psflux'], dtype=device_dtype)
psfxoffs = jnp.array(cache['xoffs'], dtype=device_dtype)
psfyoffs = jnp.array(cache['yoffs'], dtype=device_dtype)
xpx = jnp.array(cache['xpx'], dtype=device_dtype)
ypx = jnp.array(cache['ypx'], dtype=device_dtype)

Render a few of the psfs just to be sure they make sense

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 12))

idxs = np.random.choice(np.arange(psfs.shape[0]), size=4, replace=False)
for idx, ax in zip(idxs, axs.ravel()):
    ax.imshow(np.log(psfs[idx]), origin='lower', interpolation='nearest')
    ax.set_title(f"PSF {idx}, x={psfxoffs[idx]:.3f}, y={psfyoffs[idx]:.3f}")


In [ ]:
jrng, rkey = jax.random.split(jrng)

mod, training_info = NIRSpecPSFINREmulator.train(
    psfs, psfxoffs, psfyoffs, 
    batch_size, nepochs, learning_rate, 
    rkey=rkey, xypx=(xpx, ypx))

plt.semilogy(training_info['losses'])

In [ ]:
imshowkwargs = dict(origin='lower', interpolation='nearest')
for idx in np.random.choice(len(psfs), 3, replace=False):
    pred = mod.evaluate_psf(psfxoffs[idx],psfyoffs[idx], 1)
    true = psfs[idx]
    diff = pred/pred.sum() - true/true.sum()
    pdiff = (pred/pred.sum()-true/true.sum())/(true/true.sum())
    fig, axs = plt.subplots(3, 2, figsize=(10, 15))

    axs[0, 0].imshow(pred, **imshowkwargs)
    axs[1, 0].imshow(true, **imshowkwargs)
    axs[2, 0].imshow(diff, **imshowkwargs)
    axs[0, 1].imshow(jnp.log(pred), **imshowkwargs)
    axs[1, 1].imshow(jnp.log(true), **imshowkwargs)
    axs[2, 1].imshow(pdiff, **imshowkwargs, vmin=-1, vmax=1)

    axs[0, 0].set_title(f'PSF {idx}')
    axs[1, 0].set_title(f'x={psfxoffs[idx]:.4f}, y={psfyoffs[idx]:.4f}')
    axs[2, 0].set_title(f'diffmean={diff.mean():.4e}, diffstd={diff.std():.4e}')
    axs[0, 0]. set_ylabel('Predicted')
    axs[1, 0]. set_ylabel('True')
    axs[2, 0]. set_ylabel('Diff')
    axs[0, 1].set_title('log')
    axs[1, 1].set_title('log')
    axs[2, 1].set_title('pdiff')

In [ ]:
show_choices = np.random.choice(len(psfs), 3, replace=False)

In [ ]:
imshowkwargs = dict(origin='lower', interpolation='nearest')

for idx in show_choices:
    pred = mod.evaluate_psf(psfxoffs[idx],psfyoffs[idx], 1)
    true = psfs[idx]
    truen = true/np.sum(true)
    predn = pred/np.sum(pred)

    truemax = np.where(true == true.max())

    maxmax = max(predn.max(), truen.max())

    fig, axs = plt.subplots(2, 3, figsize=(16,14))
    i = axs[0,0].imshow(predn, vmin=0, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[0, 0], orientation='horizontal')
    axs[0,0].set_title(f'Predicted PSF # {idx} x={psfxoffs[idx]:.3f} y={psfyoffs[idx]:.3f}')
    i = axs[0,1].imshow(truen, vmin=0, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[0, 1], orientation='horizontal')
    axs[0,1].set_title('Normalized True PSF')
    i = axs[0,2].imshow(predn-truen, vmin=-maxmax/2, vmax=maxmax/2, **imshowkwargs)
    plt.colorbar(i, ax=axs[0, 2], orientation='horizontal')
    axs[0,2].set_title('Difference (Predicted - True)')

    i = axs[1, 0].imshow(predn, vmin=0, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[1, 0], orientation='horizontal')
    axs[1, 0].set_title('Predicted PSF')
    i = axs[1, 1].imshow(truen, vmin=0, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[1, 1], orientation='horizontal')
    axs[1, 1].set_title('Normalized True PSF')
    i = axs[1, 2].imshow(predn-truen, vmin=-maxmax, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[1, 2], orientation='horizontal')
    axs[1, 2].set_title('Difference (Predicted - True)')

    boxhsize = 5
    for ax in axs[1]:
        ax.set_ylim(truemax[0][0]-boxhsize, truemax[0][0]+boxhsize)
        ax.set_xlim(truemax[1][0]-boxhsize, truemax[1][0]+boxhsize)

In [ ]:
imshowkwargs = dict(origin='lower', interpolation='nearest')

for idx in np.random.choice(len(psfs), 3, replace=False):
    pred = mod.evaluate_psf(psfxoffs[idx],psfyoffs[idx], 1)
    true = psfs[idx]
    truen = true/np.sum(true)
    predn = pred/np.sum(pred)
    ltruen = jnp.log(truen)
    lpredn = jnp.log(predn)


    truemax = np.where(true == true.max())
    maxmax = max(lpredn.max(), ltruen.max())
    minmin = min(lpredn.min(), ltruen.min())

    fig, axs = plt.subplots(2, 3, figsize=(16, 14))
    i = axs[0,0].imshow(lpredn, vmin=minmin, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[0, 0], orientation='horizontal')
    axs[0,0].set_title(f'Predicted Log PSF # {idx} x={psfxoffs[idx]:.3f} y={psfyoffs[idx]:.3f}')
    i = axs[0,1].imshow(ltruen, vmin=minmin, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[0, 1], orientation='horizontal')
    axs[0,1].set_title('Normalized True LogPSF')
    i = axs[0,2].imshow(lpredn-ltruen, **imshowkwargs)
    plt.colorbar(i, ax=axs[0, 2], orientation='horizontal')
    axs[0,2].set_title('Difference (log Pred - log True)')

    i = axs[1, 0].imshow(lpredn, vmin=minmin, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[1, 0], orientation='horizontal')
    axs[1, 0].set_title('Predicted Log PSF')
    i = axs[1, 1].imshow(ltruen, vmin=minmin, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[1, 1], orientation='horizontal')
    axs[1, 1].set_title('Normalized True Log PSF')
    i = axs[1, 2].imshow(lpredn-ltruen, **imshowkwargs)
    plt.colorbar(i, ax=axs[1, 2], orientation='horizontal')
    axs[1, 2].set_title('Difference (log Pred - log True)')

    boxhsize = 5
    for ax in axs[1]:
        ax.set_ylim(truemax[0][0]-boxhsize, truemax[0][0]+boxhsize)
        ax.set_xlim(truemax[1][0]-boxhsize, truemax[1][0]+boxhsize)

    fig.tight_layout()

Fraction in the central slice:

In [ ]:
slc = (slice(truemax[0][0]-boxhsize, truemax[0][0]+boxhsize), slice(truemax[1][0]-boxhsize, truemax[1][0]+boxhsize))
predn[slc].sum(), truen[slc].sum(), predn[slc].sum() - truen[slc].sum()

Do a timing test to ensure with these network parameters execution doesn't take too long.

In [ ]:
inp = (psfxoffs[idx], psfyoffs[idx], 1)
st = time.time()
pred = mod.evaluate_psf(*inp).block_until_ready()
et = time.time()
print('PSF evaluation took', (et - st)*1000, 'ms')